# SmallCNN と学習ループ

**出典**　機械学習 4 分野の教科書 — 第 7 章 2-B CNN と RNN ／ 02 畳み込みとプーリング

**このノートのパス**　`ml-code/07_cnn_rnn/01_small_cnn.ipynb`

**このノートで確かめること**
Conv → BatchNorm → ReLU → MaxPool の積み方と、**テンソルの形がどう変わるか**を追う。
データ拡張を**訓練側だけ**に入れること、`zero_grad()` / `train()` / `eval()` の置き場所を確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `torch torchvision` |
| 実行時間の目安 | 約 2〜10 分（CPU） |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q torch torchvision

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 入力: 1×28×28（MNIST の手書き数字）
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # → 32×28×28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # → 32×28×28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # → 32×14×14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → 64×14×14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # → 64×7×7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),   # 出力は logits（softmax は損失関数側で適用）
        )

    def forward(self, x):
        return self.classifier(self.features(x))

### テンソルの形を、1 層ずつ実際に追う

コメントに書いた「→ 32×28×28」が本当かを確かめる。**形の勘違いは最も多いバグの原因**である。

In [4]:
m = SmallCNN().eval()
h = torch.randn(2, 1, 28, 28)
print(f"{'入力':>28}: {tuple(h.shape)}")
with torch.no_grad():
    for layer in m.features:
        h = layer(h)
        print(f"{layer.__class__.__name__:>28}: {tuple(h.shape)}")
    for layer in m.classifier:
        h = layer(h)
        print(f"{layer.__class__.__name__:>28}: {tuple(h.shape)}")

                          入力: (2, 1, 28, 28)
                      Conv2d: (2, 32, 28, 28)
                 BatchNorm2d: (2, 32, 28, 28)
                        ReLU: (2, 32, 28, 28)
                      Conv2d: (2, 32, 28, 28)
                 BatchNorm2d: (2, 32, 28, 28)
                        ReLU: (2, 32, 28, 28)
                   MaxPool2d: (2, 32, 14, 14)
                      Conv2d: (2, 64, 14, 14)
                 BatchNorm2d: (2, 64, 14, 14)
                        ReLU: (2, 64, 14, 14)
                   MaxPool2d: (2, 64, 7, 7)
                     Flatten: (2, 3136)
                     Dropout: (2, 3136)
                      Linear: (2, 128)
                        ReLU: (2, 128)
                      Linear: (2, 10)


## データの用意

MNIST は**初回だけダウンロード**する。取得できない環境では合成データに切り替える
（形だけ同じ 1×28×28。学習の手順を確かめるのが目的なので精度は意味を持たない）。

In [5]:
from torch.utils.data import DataLoader, TensorDataset

USE_MNIST = True
tf_train = tf_val = None

if USE_MNIST:
    try:
        from torchvision import datasets, transforms
        tf_train = transforms.Compose([
            transforms.RandomCrop(28, padding=4),        # 平行移動のデータ拡張
            transforms.RandomRotation(10),               # わずかに回す
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),  # MNIST の平均・標準偏差
        ])
        tf_val = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),  # ⚠️ 検証側に拡張は入れない
        ])
        train_set = datasets.MNIST("./data", train=True,  download=True, transform=tf_train)
        val_set   = datasets.MNIST("./data", train=False, download=True, transform=tf_val)
        print("MNIST:", len(train_set), len(val_set))
    except Exception as e:
        print("取得できなかったので合成データで代用:", type(e).__name__)
        USE_MNIST = False

if not USE_MNIST:
    g = torch.Generator().manual_seed(0)
    Xtr = torch.randn(6000, 1, 28, 28, generator=g)
    ytr = torch.randint(0, 10, (6000,), generator=g)
    Xtr += ytr.view(-1, 1, 1, 1) * 0.15          # ラベルごとに少しずらして学習可能にする
    Xva = torch.randn(1000, 1, 28, 28, generator=g)
    yva = torch.randint(0, 10, (1000,), generator=g)
    Xva += yva.view(-1, 1, 1, 1) * 0.15
    train_set, val_set = TensorDataset(Xtr, ytr), TensorDataset(Xva, yva)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)  # 書籍では 2。macOS では 0 で実行
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=0)

MNIST: 60000 10000


⚠️ **データ拡張は「ラベルが変わらない変形」だけを使う。**
数字を左右反転させると 2 が 5 に見えるなど意味が壊れるので、MNIST に `RandomHorizontalFlip` は入れない。

CIFAR-10（3×32×32）に替えるなら `Conv2d(1, 32, ...)` を `Conv2d(3, 32, ...)` に、
`Linear(64*7*7, ...)` を `Linear(64*8*8, ...)` に直す必要がある。

## 学習ループ

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallCNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()   # softmax + 交差エントロピーが一体になっている

EPOCHS = 2      # 書籍では 10。CPU で試すときは小さくしてよい
for epoch in range(EPOCHS):
    model.train()                      # ⚠️ dropout / BatchNorm を学習モードに
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()          # ⚠️ 勾配は累積するので毎回ゼロにする
        loss = criterion(model(x), y)
        loss.backward()                # 逆伝播（自動微分）
        optimizer.step()               # パラメータ更新

    model.eval()                       # ⚠️ 推論モードに切り替え
    correct = total = 0
    with torch.no_grad():              # 勾配を計算しない（メモリと速度の節約）
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    print(f"epoch {epoch}: val acc = {correct/total:.4f}", flush=True)

epoch 0: val acc = 0.9816


epoch 1: val acc = 0.9849


## 読み取りどころ

- **`padding=1` と `kernel_size=3` の組で、縦横のサイズが変わらない。**サイズを落とすのは `MaxPool2d` の役目、という分担になっている。
- **`Linear(64*7*7, 128)` の `7*7` は、28 を 2 回プーリングした結果。**入力サイズを変えたらここも変える必要がある——形の追跡セルが効くのはこの確認のため。
- **`optimizer.zero_grad()` を忘れると勾配が累積する。**例外は出ず、学習が進まないだけなので気づきにくい。
- **検証側に拡張を入れてはいけない。**評価のたびに違う画像を見ることになり、数字が揺れる。